# AgriSetu — Plant Disease CNN Training
## EfficientNet-Lite Fine-Tuning on PlantVillage Dataset

### Instructions:
1. **Upload the dataset** to Google Drive first:
   - Upload the entire `New Plant Diseases Dataset(Augmented)` folder to `My Drive/agrisetu/`
   - Or download directly in Colab using Kaggle CLI (Cell 2)

2. **Set runtime to GPU:**
   - Go to Runtime → Change runtime type → **T4 GPU**

3. **Run all cells in order** (Shift+Enter)

4. **After training:** Download the saved model weights from `models/disease_model/` folder
   - Right-click `disease_model_best.pth` → Download
   - Place it in `agrisetu-backend/models/disease_model/`

### Expected Output:
- Training: 15 epochs, ~10-15 min on T4 GPU
- Target accuracy: >90% on validation set
- Saved: `disease_model_best.pth` + `class_names.json`

## Cell 1 — Install Dependencies & Mount Google Drive

In [ ]:
!pip install -q timm tqdm scikit-learn

import torch
import torchvision
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create working directory
import os
WORK_DIR = '/content/drive/MyDrive/agrisetu'
os.makedirs(f'{WORK_DIR}/models/disease_model', exist_ok=True)
print(f"Working directory: {WORK_DIR}")

## Cell 2 — Download Dataset (Choose ONE option)

**Option A:** If dataset is already in Google Drive, skip this cell.

**Option B:** Download from Kaggle (requires kaggle.json).

In [ ]:
# OPTION B: Download from Kaggle
# Uncomment and run this cell if dataset is NOT in Google Drive

# !pip install -q kaggle
# from google.colab import files
# files.upload()  # Upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content/dataset
# !unzip -q /content/dataset/new-plant-diseases-dataset.zip -d /content/dataset/
# DATASET_DIR = '/content/dataset/New Plant Diseases Dataset(Augmented)'
# print("Dataset downloaded!")

## Cell 3 — Configure Dataset Paths

In [ ]:
import os

# If dataset is in Google Drive, use this path:
DATASET_DIR = f'{WORK_DIR}/New Plant Diseases Dataset(Augmented)'

# If dataset was downloaded via Kaggle, use this instead:
# DATASET_DIR = '/content/dataset/New Plant Diseases Dataset(Augmented)'

TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
VAL_DIR = os.path.join(DATASET_DIR, 'valid')

# Verify paths exist
assert os.path.exists(TRAIN_DIR), f"Train dir not found: {TRAIN_DIR}"
assert os.path.exists(VAL_DIR), f"Val dir not found: {VAL_DIR}"

train_classes = sorted(os.listdir(TRAIN_DIR))
num_classes = len(train_classes)
print(f"Number of classes: {num_classes}")
print(f"Classes: {train_classes}")

# Count images
total_train = sum(len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in train_classes)
total_val = sum(len(os.listdir(os.path.join(VAL_DIR, c))) for c in train_classes)
print(f"Training images: {total_train}")
print(f"Validation images: {total_val}")

## Cell 4 — Create Data Loaders with Augmentation

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Image size for EfficientNet-Lite (224x224)
IMG_SIZE = 224
BATCH_SIZE = 32

# Training transforms (with augmentation)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation transforms (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Get class names
class_names = train_dataset.classes
print(f"Classes loaded: {len(class_names)}")
print(f"Sample classes: {class_names[:5]}")

# Get a batch to verify
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")

## Cell 5 — Define Model (EfficientNet-Lite with Custom Head)

In [ ]:
import torch.nn as nn
import timm

# Load pretrained EfficientNet-Lite0
model = timm.create_model('efficientnet_lite0', pretrained=True, num_classes=num_classes)

# Freeze all layers except the last 30 (fine-tune top layers)
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the classifier head and last few blocks
for param in model.classifier.parameters():
    param.requires_grad = True

# Unfreeze last 2 blocks of the model
blocks = list(model.children())
if hasattr(model, 'blocks'):
    for block in model.blocks[-2:]:
        for param in block.parameters():
            param.requires_grad = True

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Model on: {device}")

## Cell 6 — Training Configuration

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

# Training config
NUM_EPOCHS = 15
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer (only trainable parameters)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print(f"Epochs: {NUM_EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Device: {device}")

## Cell 7 — Training Loop

In [ ]:
import time
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    start_time = time.time()

    # ---- Training Phase ----
    model.train()
    train_loss = 0.0
    train_preds = []
    train_labels = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    train_loss /= len(train_dataset)
    train_acc = accuracy_score(train_labels, train_preds)

    # ---- Validation Phase ----
    model.eval()
    val_loss = 0.0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_dataset)
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds, average='macro')

    # Update scheduler
    scheduler.step()

    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - start_time

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} ({elapsed:.1f}s)")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val F1: {val_f1:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'num_classes': num_classes,
            'class_names': class_names,
            'val_acc': val_acc,
            'val_f1': val_f1,
            'epoch': epoch + 1,
        }, f'{WORK_DIR}/models/disease_model/disease_model_best.pth')
        print(f"  ✓ Best model saved (val_acc: {val_acc:.4f})")

print(f"\n{'='*50}")
print(f"Training complete! Best validation accuracy: {best_val_acc:.4f}")

## Cell 8 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
ax1.plot(history['train_loss'], label='Train Loss', marker='o')
ax1.plot(history['val_loss'], label='Val Loss', marker='o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(history['train_acc'], label='Train Acc', marker='o')
ax2.plot(history['val_acc'], label='Val Acc', marker='o')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training & Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/models/disease_model/training_curves.png', dpi=150)
plt.show()
print("Training curves saved!")

## Cell 9 — Detailed Evaluation + Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Load best model
checkpoint = torch.load(f'{WORK_DIR}/models/disease_model/disease_model_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Run full validation
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Classification report
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(all_labels, all_preds, target_names=class_names))

print(f"\nBest model val accuracy: {checkpoint['val_acc']:.4f}")
print(f"Best model val F1: {checkpoint['val_f1']:.4f}")

## Cell 10 — Save Class Names JSON (for backend)

In [ ]:
import json

# Save class names as JSON for the backend
class_names_dict = {i: name for i, name in enumerate(class_names)}
with open(f'{WORK_DIR}/models/disease_model/class_names.json', 'w') as f:
    json.dump(class_names_dict, f, indent=2)

print("Class names saved to class_names.json")
print(json.dumps(class_names_dict, indent=2))

## Cell 11 — Test Inference on Sample Images

In [ ]:
from PIL import Image
import random

# Test on random validation images
def predict_image(image_path, model, transform):
    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.nn.functional.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)

    return class_names[predicted.item()], confidence.item()

# Test 5 random images
print("Sample predictions:")
print("=" * 60)
for cls in random.sample(class_names, min(5, len(class_names))):
    cls_dir = os.path.join(VAL_DIR, cls)
    if os.listdir(cls_dir):
        img_path = os.path.join(cls_dir, os.listdir(cls_dir)[0])
        pred, conf = predict_image(img_path, model, val_transform)
        true_label = cls
        status = "✓" if pred == true_label else "✗"
        print(f"{status} True: {true_label}")
        print(f"  Pred: {pred} (confidence: {conf:.4f})")
        print()

## Cell 12 — Export Model for Backend

This cell creates a lightweight version of the model for deployment.

In [ ]:
# Export model in a format ready for the backend
# The backend will load this with torch.load()

export_path = f'{WORK_DIR}/models/disease_model/disease_model_best.pth'
checkpoint = torch.load(export_path)

# Verify the saved model
print("Saved model info:")
print(f"  Num classes: {checkpoint['num_classes']}")
print(f"  Val accuracy: {checkpoint['val_acc']:.4f}")
print(f"  Val F1: {checkpoint['val_f1']:.4f}")
print(f"  Epoch: {checkpoint['epoch']}")
print(f"  Class names: {len(checkpoint['class_names'])} classes")

# Files to download:
print(f"\n{'='*60}")
print("DOWNLOAD THESE FILES:")
print(f"  1. {WORK_DIR}/models/disease_model/disease_model_best.pth")
print(f"  2. {WORK_DIR}/models/disease_model/class_names.json")
print(f"\nPlace them in: agrisetu-backend/models/disease_model/")
print(f"{'='*60}")